# All Three Patches in One Process

This notebook applies all three patches simultaneously in a single Ruby session — no `require_relative`, everything inline.

| Patch | Intercept point | Side effect blocked |
|---|---|---|
| AR | `before_save` on `ActiveRecord::Base` | DB write (Alice, direct update) |
| GraphQL | `resolve` on `UpdateUserAgeMutation` | DB write (Bob, via mutation) |
| Karafka | `produce_sync` on `WaterDrop::Producer` | Kafka message delivery |

## Setup — ActiveRecord + GraphQL schema + WaterDrop producer

> **Full reset only:** restart the kernel and re-run all cells — GraphQL schema classes and patches cannot be safely re-applied mid-session.

In [ ]:
require "active_record"
require "graphql"
require "waterdrop"

Object.send(:remove_const, :DB_PATH) if defined?(DB_PATH)
DB_PATH = "/lab/ruby/notebooks/all_demo.db"

# -- ActiveRecord --------------------------------------------------------------
ActiveRecord::Base.establish_connection(adapter: "sqlite3", database: DB_PATH)
ActiveRecord::Schema.verbose = false
ActiveRecord::Schema.define do
  create_table :users, force: :cascade do |t|
    t.string  :name
    t.string  :email
    t.integer :age
    t.timestamps
  end
end

class User < ActiveRecord::Base
  self.table_name = "users"
end

User.create!(name: "Alice", email: "alice@example.com", age: 30)
User.create!(name: "Bob",   email: "bob@example.com",   age: 25)

# -- GraphQL schema with a mutation --------------------------------------------
module Types
  class UserType < GraphQL::Schema::Object
    field :id,    ID,      null: false
    field :name,  String,  null: true
    field :age,   Integer, null: true
  end

  class QueryType < GraphQL::Schema::Object
    field :users, [UserType], null: false
    def users = User.all
  end

  class UpdateUserAgeMutation < GraphQL::Schema::Mutation
    argument :id,  ID,      required: true
    argument :age, Integer, required: true
    field :user, UserType, null: true

    def resolve(id:, age:)
      user = User.find(id)
      user.update!(age: age)
      { user: user.reload }
    end
  end

  class MutationType < GraphQL::Schema::Object
    field :update_user_age, mutation: UpdateUserAgeMutation
  end
end

class AppSchema < GraphQL::Schema
  query    Types::QueryType
  mutation Types::MutationType
end

# -- WaterDrop producer --------------------------------------------------------
PRODUCER.close rescue nil if defined?(PRODUCER)
Object.send(:remove_const, :PRODUCER) if defined?(PRODUCER)
PRODUCER = WaterDrop::Producer.new do |config|
  config.deliver = false
  config.kafka   = { "bootstrap.servers": "localhost:9092" }
  config.logger  = Logger.new(File::NULL)
end

User.all.each { |u| puts "  #{u.name}: age=#{u.age}" }
puts "Setup complete"

## Shared context objects

`RollbackContext` collects both AR snapshots and GraphQL mutation snapshots in one place.  
`CaptureContext` collects Kafka messages.

In [ ]:
module RollbackContext
  @mode      = :real
  @snapshots = []

  def self.dry_run!  = (@mode = :dry;  @snapshots = [])
  def self.real_run! = (@mode = :real)
  def self.dry_run?  = (@mode == :dry)

  def self.add_snapshot(table:, record_id:, operation:, before:)
    @snapshots << { table: table, record_id: record_id, operation: operation, before: before }
  end

  def self.snapshots = @snapshots.dup

  def self.rollback!
    @snapshots.each do |snap|
      next unless snap[:operation] == :update
      klass = ActiveRecord::Base.descendants.find { |k| k.table_name == snap[:table] }
      klass&.find(snap[:record_id])
            &.update_columns(snap[:before].except('id'))
    end
    @snapshots = []; @mode = :real
  end
end

module CaptureContext
  @capturing = false
  @messages  = []  # Kafka messages
  @mutations = []  # GraphQL mutations

  def self.start!
    @capturing = true; @messages = []; @mutations = []
  end

  def self.stop!      = (@capturing = false)
  def self.capturing? = @capturing

  def self.add(message)
    @messages << { topic: message[:topic], payload: message[:payload] }
  end

  def self.add_mutation(query:, variables: nil)
    @mutations << { query: query, variables: variables }
  end

  def self.messages  = @messages.dup
  def self.mutations = @mutations.dup
end

puts 'Context objects ready'

## The three patches

All three are applied in a single cell.

**AR** has the full dry-run / real-run / rollback cycle.
**GraphQL and Kafka** are capture-only: the mutation/message is recorded and the original call is never made, mirroring how both send to external systems whose side effects cannot be cleanly reversed here.

In [ ]:
# NOTE: run this cell only once per kernel session.
# Re-running adds duplicate AR callbacks and breaks the alias_method chain for GraphQL/Kafka.
# To reset, restart the kernel and re-run all cells from the top.

# -- Patch 1: ActiveRecord::Base.class_eval -----------------------------------
ActiveRecord::Base.class_eval do
  before_save    :capture_before_save,    if: -> { RollbackContext.dry_run? }
  before_destroy :capture_before_destroy, if: -> { RollbackContext.dry_run? }

  private

  def capture_before_save
    unless new_record?
      before = self.class.column_names.to_h do |col|
        [col, attribute_changed?(col) ? attribute_was(col) : read_attribute(col)]
      end
      RollbackContext.add_snapshot(table: self.class.table_name, record_id: id,
                                   operation: :update, before: before)
    end
    throw :abort
  end

  def capture_before_destroy
    RollbackContext.add_snapshot(table: self.class.table_name, record_id: id,
                                 operation: :destroy, before: attributes.dup)
    throw :abort
  end
end

# -- Patch 2: AppSchema.singleton_class.class_eval ----------------------------
# Capture-only: mutation is recorded, schema never executes.
AppSchema.singleton_class.class_eval do
  alias_method :original_execute, :execute

  def execute(query_str = nil, **kwargs)
    if CaptureContext.capturing?
      CaptureContext.add_mutation(query: query_str, variables: kwargs[:variables])
      return {}
    end
    original_execute(query_str, **kwargs)
  end
end

# -- Patch 3: WaterDrop::Producer.class_eval ----------------------------------
WaterDrop::Producer.class_eval do
  alias_method :original_produce_sync, :produce_sync

  def produce_sync(**kwargs)
    if CaptureContext.capturing?
      CaptureContext.add(kwargs)
      return
    end
    original_produce_sync(**kwargs)
  end
end

puts 'All three patches applied'

## Demo data

In [ ]:
Object.send(:remove_const, :EVENTS) if defined?(EVENTS)
Object.send(:remove_const, :BOB_MUTATION) if defined?(BOB_MUTATION)
EVENTS = [
  { topic: 'user_events', payload: { event: 'user_created', name: 'Alice' }.to_json },
  { topic: 'user_events', payload: { event: 'user_updated', name: 'Bob'   }.to_json }
]

BOB_MUTATION = 'mutation { updateUserAge(id: "2", age: 88) { user { name age } } }'

## Phase 1 — Capture / Dry Run

All three patches intercept:
- **AR**: Alice's save is aborted, snapshot captured for Phase 3 rollback.
- **GraphQL**: Bob's mutation string is captured, schema never executes.
- **Kafka**: messages are captured, broker never contacted.

In [ ]:
RollbackContext.dry_run!
CaptureContext.start!

result = User.find_by(name: 'Alice').update(age: 99)
puts "AR      -> alice.update(age: 99) returned #{result.inspect}  (aborted)"

AppSchema.execute(BOB_MUTATION)
puts "GraphQL -> #{CaptureContext.mutations.size} mutation(s) captured, schema never executed"

EVENTS.each { |e| PRODUCER.produce_sync(**e) }
puts "Kafka   -> #{CaptureContext.messages.size} message(s) captured, Kafka never touched"

puts
puts "DB  -> Alice: age=#{User.find_by(name: 'Alice').age}  Bob: age=#{User.find_by(name: 'Bob').age}"
puts "AR snapshot:      #{RollbackContext.snapshots.first&.slice(:table, :record_id, :operation)}"
puts "GraphQL captured: #{CaptureContext.mutations.size} mutation(s)"
puts "Kafka captured:   #{CaptureContext.messages.map { |m| m[:topic] }}"

## Phase 2 — Real Run

Guards are off — all three side effects execute.

In [ ]:
RollbackContext.real_run!
CaptureContext.stop!

User.find_by(name: 'Alice').update!(age: 99)
puts 'AR      -> alice.update!(age: 99) saved'

gql  = AppSchema.execute(BOB_MUTATION)
data = gql['data']&.dig('updateUserAge', 'user')
puts "GraphQL -> #{data.inspect}"

EVENTS.each { |e| PRODUCER.produce_sync(**e) }
puts "Kafka   -> #{EVENTS.size} message(s) dispatched"

puts
puts "DB  -> Alice: age=#{User.find_by(name: 'Alice').age}  Bob: age=#{User.find_by(name: 'Bob').age}"

## Phase 3 — Rollback (AR only)

AR restores Alice from the Phase 1 snapshot.
GraphQL and Kafka are capture-only — Bob stays at age=88 after Phase 2.

In [ ]:
RollbackContext.rollback!

puts "DB  -> Alice: age=#{User.find_by(name: 'Alice').age}  Bob: age=#{User.find_by(name: 'Bob').age}"
puts "       (Alice restored from snapshot; Bob unchanged -- GraphQL is capture-only)"
puts 'GraphQL -> capture-only, no rollback'
puts 'Kafka   -> capture-only, no rollback'

## Cleanup

Run the cell below when you are done to close the WaterDrop producer cleanly.

In [ ]:
PRODUCER.close
puts "Producer closed."